In [21]:
import pandas as pd

# =========================
# 1. LOAD CSV FILES
# =========================
mri_csv_path = "mri.csv"
pet_csv_path = "pet.csv"

mri_df = pd.read_csv(r"C:\Users\fhabi\Desktop\DataSet\MRI1_4_12_2026.csv")
pet_df = pd.read_csv(r"C:\Users\fhabi\Desktop\DataSet\PET2_4_12_2026.csv")

# =========================
# 2. CLEAN COLUMN NAMES (optional but important)
# =========================
mri_df.columns = mri_df.columns.str.strip()
pet_df.columns = pet_df.columns.str.strip()

# =========================
# 3. FILTER MRI (ONLY MP-RAGE)
# =========================
mri_df = mri_df[mri_df["Description"].str.contains("MP-RAGE", na=False)]

# =========================
# 4. FILTER PET (FDG or AV45)
# =========================
pet_df = pet_df[
    pet_df["Description"].str.contains("FDG", na=False)  #|AV45|Florbetapir
]

# =========================
# 5. SELECT IMPORTANT COLUMNS
# =========================
mri_df = mri_df[["Subject", "Visit", "Acq Date", "Description"]]
pet_df = pet_df[["Subject", "Visit", "Acq Date", "Description"]]

# =========================
# 6. CONVERT DATE FORMAT
# =========================
mri_df["Acq Date"] = pd.to_datetime(mri_df["Acq Date"])
pet_df["Acq Date"] = pd.to_datetime(pet_df["Acq Date"])

# =========================
# 7. RENAME COLUMNS
# =========================
mri_df = mri_df.rename(columns={
    "Acq Date": "MRI_Date",
    "Description": "MRI_Type"
})

pet_df = pet_df.rename(columns={
    "Acq Date": "PET_Date",
    "Description": "PET_Type"
})

# =========================
# 8. MERGE BY SUBJECT + VISIT
# =========================
merged = pd.merge(
    mri_df,
    pet_df,
    on=["Subject", "Visit"],
    how="inner"
)

# =========================
# 9. COMPUTE DATE DIFFERENCE
# =========================
merged["Date_Diff_Days"] = abs(
    (merged["MRI_Date"] - merged["PET_Date"]).dt.days
)

# =========================
# 10. FILTER CLOSE PAIRS (IMPORTANT)
# =========================
# keep only pairs within 90 days (standard ADNI practice)
paired = merged[merged["Date_Diff_Days"] <= 180]

# =========================
# 11. FINAL CLEAN DATASET
# =========================
paired = paired.sort_values(["Subject", "Visit"])

# =========================
# 12. SAVE OUTPUT
# =========================
paired.to_csv("ADNI_MRI_PET_Pairs.csv", index=False)

print("DONE ✅ Pairing completed!")
print("Total pairs:", len(paired))
print(paired.head())

DONE ✅ Pairing completed!
Total pairs: 230
        Subject Visit   MRI_Date MRI_Type   PET_Date  \
229  002_S_0685   m48 2010-07-15  MP-RAGE 2010-07-27   
228  002_S_0729   m48 2010-07-22  MP-RAGE 2010-07-29   
227  002_S_1155   m48 2011-01-06  MP-RAGE 2011-01-07   
224  005_S_0221   m06 2006-10-06  MP-RAGE 2006-10-06   
225  005_S_0221   m12 2007-03-21  MP-RAGE 2007-03-20   

                            PET_Type  Date_Diff_Days  
229          ADNI Brain PET: Raw FDG              12  
228          ADNI Brain PET: Raw FDG               7  
227          ADNI Brain PET: Raw FDG               1  
224             30 min 3D FDG 4i/16s               0  
225  30 min 3D FDG - Iter(Brain Mode               1  


In [22]:
subjects = paired["Subject"].unique()
print(subjects)
pd.DataFrame(subjects, columns=["Subject"]).to_csv(r"C:\Users\fhabi\Desktop\DataSet\subjects.csv", index=False)

['002_S_0685' '002_S_0729' '002_S_1155' '005_S_0221' '005_S_0222'
 '005_S_0223' '005_S_0546' '005_S_0553' '005_S_0602' '005_S_0610'
 '005_S_0929' '005_S_1224' '005_S_1341' '006_S_0498' '006_S_0675'
 '006_S_1130' '007_S_0101' '007_S_0128' '007_S_0293' '007_S_0316'
 '007_S_0344' '007_S_0414' '007_S_0698' '007_S_1339' '009_S_0842'
 '009_S_1030' '014_S_0519' '014_S_0520' '014_S_0548' '014_S_0563'
 '014_S_0658' '021_S_0343' '021_S_0626' '021_S_0984' '024_S_0985'
 '024_S_1063' '024_S_1171' '024_S_1307' '024_S_1393' '024_S_1400'
 '027_S_0074' '027_S_0120' '027_S_0307' '027_S_0408' '027_S_1045'
 '029_S_0845' '029_S_1218' '033_S_0734' '033_S_0741' '033_S_0906'
 '033_S_0920' '033_S_0923' '033_S_1016' '033_S_1098' '037_S_0150'
 '037_S_0327' '037_S_0377' '037_S_0454' '037_S_0467' '037_S_0552'
 '037_S_0566' '037_S_1078' '037_S_1421' '052_S_0671' '052_S_0989'
 '052_S_1346' '052_S_1352' '053_S_0919' '094_S_0489' '094_S_0526'
 '094_S_0531' '094_S_1090' '094_S_1164' '094_S_1188' '094_S_1314'
 '094_S_13